# Ders 3B — RandomForest ile Sampling Senaryoları

Step 5 sonucuna göre iki veri setinde de RandomForest ile devam edilir.

Bu notebookta aynı train/test split korunur. Oversampling ve undersampling yalnızca train set üzerinde uygulanır; test set doğal dağılımında kalır.


## 1. Paket kontrolü

Bu scriptte gerçekten kullanılacak paketler kontrol edilir. Gereksiz paket import edilmez.

Bu dosyada yalnızca RandomForest ve manuel sampling kullanılır; `imbalanced-learn` gibi ek paket gerekmez.


In [ ]:
import sys
# Mevcut Python ortamında pip komutunu çalıştırmak için kullanılır.
import subprocess
# Eksik paketleri notebook içinden kurmak için kullanılır.
import importlib.util
# Bir paketin kurulu olup olmadığını kontrol etmek için kullanılır.

def install_if_missing(import_name, pip_name=None):
    """Eksik paket varsa kurar; paket zaten varsa işlem yapmaz."""
    pip_name = pip_name or import_name
    # pip adı verilmezse import adı paket adı olarak kullanılır.
    if importlib.util.find_spec(import_name) is None:
        # Paket kurulu değilse pip kurulumu yapılır.
        print(f"[INSTALL] {pip_name}")
        # Hangi paketin kurulacağı ekrana yazdırılır.
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pip_name])
        # Paket sessiz modda kurulur.

required_packages = [
    ("pandas", "pandas"),  # CSV okuma ve tablo işlemleri için kullanılır.
    ("numpy", "numpy"),  # Sayısal matris ve vektör işlemleri için kullanılır.
    ("matplotlib", "matplotlib"),  # Grafik çizimleri için kullanılır.
    ("sklearn", "scikit-learn"),  # Makine öğrenmesi modelleri ve metrikleri için kullanılır.
]
# Bu scriptte gerçekten kullanılan paketler listelenir.

for import_name, pip_name in required_packages:
    # Paketler tek tek kontrol edilir.
    install_if_missing(import_name, pip_name)
    # Eksik paket varsa kurulur.

print("Paket kontrolü tamamlandı.")
# Paket kontrolünün bittiği yazdırılır.

## 2. Importlar ve genel ayarlar

Bu hücrede yalnızca bu scriptte kullanılacak paketler ve sabitler çağırılır.

In [ ]:
from pathlib import Path
# Dosya ve klasör yollarını güvenli şekilde yönetmek için kullanılır.
import warnings
# Gereksiz uyarıları kontrol etmek için kullanılır.
warnings.filterwarnings("ignore")
# Notebook çıktısını sade tutmak için uyarılar gizlenir.

import numpy as np
# Feature matrisleri, sampling indexleri ve skor düzenleme işlemleri için kullanılır.
import pandas as pd
# Feature CSV dosyalarını okumak ve sonuç tablolarını kaydetmek için kullanılır.
import matplotlib.pyplot as plt
# Sampling sonuçlarını bar chart olarak çizmek için kullanılır.

from sklearn.model_selection import train_test_split
# Aynı split mantığıyla train/test ayrımı yapmak için kullanılır.
from sklearn.metrics import accuracy_score, balanced_accuracy_score, average_precision_score, f1_score, precision_score, recall_score, roc_auc_score, confusion_matrix
# Sampling sonuçlarını ölçmek için gerekli metrikler çağırılır.
from sklearn.ensemble import RandomForestClassifier
# Bu notebookta yalnızca RandomForest modeli kullanılacaktır.
from sklearn.impute import SimpleImputer
# Eksik değerleri train setten öğrenilen median değerlerle doldurmak için kullanılır.

DATASETS = {
    # İki veri setinin kısa isimleri ve GitHub raw feature dosyaları burada tanımlanır.
    "ERa_BLA_assay": {
        "short_name": "ERα BLA",
        "model_prefix": "model_ERa_BLA",
        "feature_url": "https://raw.githubusercontent.com/MOL-FEST/MOL_FEST_2026/main/molfest_outputs/01_feature_store/model_ERa_BLA_features.csv",
        "raw_url": "https://raw.githubusercontent.com/MOL-FEST/MOL_FEST_2026/main/Train_2_1_A14_A17_ERa_BLA_agonist_antagonist.csv",
    },
    "ERa_LUC_VM7_assay": {
        "short_name": "ERα LUC VM7",
        "model_prefix": "model_ERa_LUC_VM7",
        "feature_url": "https://raw.githubusercontent.com/MOL-FEST/MOL_FEST_2026/main/molfest_outputs/01_feature_store/model_ERa_LUC_VM7_features.csv",
        "raw_url": "https://raw.githubusercontent.com/MOL-FEST/MOL_FEST_2026/main/Train_2_2_A15_A18_ERa_LUC_VM7_agonist_antagonist.csv",
    },
}

RANDOM_STATE = 42
# Train/test ayrımı, RandomForest ve sampling için sabit random seed belirlenir.
TEST_SIZE = 0.20
# Test set oranı %20 olarak belirlenir.

OUTPUT_ROOT = Path("molfest_outputs")
# Bu scriptin çıktılarının kaydedileceği ana klasör belirlenir.
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
# Çıktı klasörü yoksa oluşturulur.

print("Ayarlar hazır.")
# Ayar hücresinin tamamlandığı yazdırılır.
print("Feature CSV klasörü:", "https://raw.githubusercontent.com/MOL-FEST/MOL_FEST_2026/main/molfest_outputs/01_feature_store")
# Hazır feature dosyalarının okunacağı GitHub raw klasörü gösterilir.


## 3. Yardımcı fonksiyonlar

Bu fonksiyonlar notebook içinde görünür durumdadır ve sadece bu scriptte ihtiyaç duyulan işlemleri kapsar.

In [ ]:
def show_table(df, n=50, title=None):
    """Sonuç tablosunu Colab'da tablo olarak, terminalde metin olarak gösterir."""
    if title:
        # Tablo başlığı varsa önce başlık yazdırılır.
        print("\n" + title)
    try:
        # Colab/Jupyter ortamında display daha okunabilir tablo üretir.
        display(df.head(n))
    except NameError:
        # Terminalde display yoksa tablo metin olarak yazdırılır.
        print(df.head(n).to_string(index=False))


def load_feature_table(dataset_key):
    """Seçilen veri setinin hazır feature CSV dosyasını GitHub raw linkinden okur."""
    info = DATASETS[dataset_key]
    # Veri setine ait URL ve kısa isim bilgisi alınır.
    df = pd.read_csv(info["feature_url"])
    # Feature CSV doğrudan GitHub raw linkinden okunur.
    print(f"\n{dataset_key} okundu: {df.shape[0]} satır, {df.shape[1]} kolon")
    # Okunan tablonun boyutu ekrana yazdırılır.
    return df
    # Okunan tablo modelleme adımlarına gönderilir.


def feature_columns(df, feature_set):
    """İstenen feature ailesine ait kolonları seçer."""
    prefix_map = {
        # Feature set adları CSV kolon prefixleriyle eşleştirilir.
        "morgan": ["Morgan_"],
        "maccs": ["MACCS_"],
        "rdkit": ["RDKit_"],
        "avalon": ["Avalon_"],
        "maccs_morgan": ["MACCS_", "Morgan_"],
        "maccs_rdkit": ["MACCS_", "RDKit_"],
        "morgan_rdkit": ["Morgan_", "RDKit_"],
        "all_available": ["Morgan_", "MACCS_", "RDKit_", "Avalon_"],
    }
    prefixes = prefix_map[feature_set]
    # İstenen feature setinin kolon prefixleri alınır.
    cols = [c for c in df.columns if any(c.startswith(p) for p in prefixes)]
    # Prefix ile eşleşen feature kolonları seçilir.
    if not cols:
        # Hiç kolon bulunmazsa erken ve anlaşılır hata verilir.
        raise ValueError(f"{feature_set} için feature kolonu bulunamadı.")
    return cols
    # Seçilen feature kolon isimleri döndürülür.


def make_xy(df, cols):
    """Feature kolonlarından X matrisi ve Target kolonundan y vektörü üretir."""
    X = (
        df[cols]
        # Sadece seçilen feature kolonları alınır.
        .apply(pd.to_numeric, errors="coerce")
        # Sayısal olmayan değerler güvenli şekilde NaN yapılır.
        .replace([np.inf, -np.inf], np.nan)
        # Sonsuz değerler NaN'a çevrilir.
        .to_numpy(dtype=np.float32)
        # sklearn modelleri için numpy matrise çevrilir.
    )
    y = df["Target"].astype(int).to_numpy()
    # Binary target kolonu integer numpy vektörüne çevrilir.
    return X, y
    # Modelleme için X ve y döndürülür.


def split_data(df, cols):
    """Sınıf oranını koruyarak train/test ayrımı yapar."""
    X, y = make_xy(df, cols)
    # Seçilen featurelardan X ve y hazırlanır.
    return train_test_split(X, y, df, test_size=TEST_SIZE, stratify=y, random_state=RANDOM_STATE)
    # Stratify ile sınıf oranı train/test içinde korunur.


def impute_train_test(X_train, X_test):
    """Eksik değerleri yalnızca train setten öğrenilen median değerlerle doldurur."""
    imputer = SimpleImputer(strategy="median")
    # SimpleImputer her feature için median değeri train setten öğrenir.
    X_train_imputed = imputer.fit_transform(X_train)
    # Imputer sadece train set üzerinde fit edilir; test bilgisi train'e sızmaz.
    X_test_imputed = imputer.transform(X_test)
    # Test set aynı imputer ile dönüştürülür.
    return X_train_imputed, X_test_imputed
    # Model eğitiminde kullanılacak temiz train/test matrisleri döndürülür.


def class1_score(model, X):
    """Modelden class 1 için skor veya olasılık üretir."""
    if hasattr(model, "predict_proba"):
        # Model olasılık üretebiliyorsa class 1 olasılığı alınır.
        score = model.predict_proba(X)[:, 1]
        # Class 1 olasılıkları alınır.
    elif hasattr(model, "decision_function"):
        # Olasılık yoksa karar fonksiyonu skoru kullanılır.
        score = model.decision_function(X)
        # Karar fonksiyonu skoru alınır.
    else:
        score = model.predict(X).astype(float)
        # Son seçenek olarak sınıf tahmini skor gibi kullanılır.

    score = np.asarray(score, dtype=float)
    # Skorlar numpy array formatına çevrilir.
    score = np.nan_to_num(score, nan=0.5, posinf=1.0, neginf=0.0)
    # ROC hesabı bozulmasın diye NaN/inf skorlar güvenli değerlere çekilir.
    return score
    # Temizlenmiş class 1 skoru döndürülür.


def metric_dict(y_true, y_pred, y_score):
    """Test tahminlerinden temel sınıflandırma metriklerini hesaplar."""
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0, 1]).ravel()
    # Confusion matrix değerleri ayrı ayrı alınır.
    specificity = tn / (tn + fp) if (tn + fp) else np.nan
    # Specificity = TN / (TN + FP) olarak hesaplanır.
    return {
        "ROC": roc_auc_score(y_true, y_score),
        # ROC-AUC, pozitif ve negatif sınıf ayrım gücünü ölçer.
        "AP": average_precision_score(y_true, y_score),
        # AP, precision-recall eğrisi altındaki ortalama performanstır.
        "F1": f1_score(y_true, y_pred, zero_division=0),
        # F1, precision ve recall dengesini özetler.
        "Accuracy": accuracy_score(y_true, y_pred),
        # Accuracy, toplam doğru sınıflandırma oranıdır.
        "BalancedAccuracy": balanced_accuracy_score(y_true, y_pred),
        # Balanced accuracy, sınıf dengesizliğinde daha adil bir doğruluk ölçüsüdür.
        "Recall": recall_score(y_true, y_pred, zero_division=0),
        # Recall = TP / (TP + FN) olarak hesaplanır.
        "Specificity": specificity,
        # Specificity = TN / (TN + FP) olarak hesaplanır.
        "Precision": precision_score(y_true, y_pred, zero_division=0),
        # Precision = TP / (TP + FP) olarak hesaplanır.
        "TN": int(tn),
        "FP": int(fp),
        "FN": int(fn),
        "TP": int(tp),
    }


def plot_metric_bar(df, label_col, metric, title, out_file=None, top_n=None):
    """Performans sonuçlarını yatay bar chart olarak çizer."""
    plot_df = df.sort_values(metric, ascending=False).copy()
    # En iyi sonuçlar üstte olacak şekilde metrik değerine göre sıralanır.
    if top_n is not None:
        # Çok uzun tablolar için ilk n sonuç çizilebilir.
        plot_df = plot_df.head(top_n)
    plot_df = plot_df.sort_values(metric, ascending=True)
    # Yatay bar chartta en iyi değer üstte görünsün diye çizim öncesi ters sıralanır.

    plt.figure(figsize=(9, max(4, 0.35 * len(plot_df))))
    # Grafik boyutu satır sayısına göre ayarlanır.
    plt.barh(plot_df[label_col].astype(str), plot_df[metric])
    # Her model/feature set için metrik değeri yatay bar olarak çizilir.
    plt.xlabel(metric)
    # X eksenine hangi metrik çizildiği yazılır.
    if metric == "ROC":
        # ROC-AUC farkları daha net görünsün diye ROC grafikleri 0.60'tan başlatılır.
        plt.xlim(left=0.60)
    plt.title(title)
    # Grafiğe açıklayıcı başlık eklenir.
    plt.tight_layout()
    # Etiketlerin kesilmemesi için yerleşim düzenlenir.
    if out_file:
        # Dosya yolu verilmişse grafik kaydedilir.
        Path(out_file).parent.mkdir(parents=True, exist_ok=True)
        # Kayıt klasörü yoksa oluşturulur.
        plt.savefig(out_file, dpi=300, bbox_inches="tight")
        # Grafik yüksek çözünürlüklü PNG olarak kaydedilir.
    plt.show()
    # Grafik ekranda gösterilir.


def add_gain(current_df, previous_df, current_name, previous_name):
    """Bir önceki adıma göre ROC/AP/F1 farkını ve yüzde değişimi hesaplar."""
    rows = []
    # Hesaplanan karşılaştırma satırları burada biriktirilir.
    for _, current in current_df.iterrows():
        # Mevcut adımın her veri seti sonucu tek tek dolaşılır.
        dataset = current["Dataset"]
        # Karşılaştırılacak veri seti adı alınır.
        prev = previous_df[previous_df["Dataset"] == dataset].sort_values("ROC", ascending=False).iloc[0]
        # Aynı veri setinin önceki adımdaki en iyi sonucu alınır.
        row = current.to_dict()
        # Mevcut sonuç satırı sözlüğe çevrilir.
        row["CurrentStep"] = current_name
        # Mevcut adım adı eklenir.
        row["ComparedWith"] = previous_name
        # Karşılaştırılan önceki adım adı eklenir.
        row["Previous_ROC"] = prev["ROC"]
        # Önceki ROC değeri tabloya eklenir.
        row["ROC_Delta"] = current["ROC"] - prev["ROC"]
        # Mutlak ROC farkı hesaplanır.
        row["ROC_Gain_%"] = 100 * (current["ROC"] - prev["ROC"]) / abs(prev["ROC"]) if abs(prev["ROC"]) > 1e-12 else np.nan
        # ROC yüzde değişimi hesaplanır.
        row["Previous_AP"] = prev["AP"]
        # Önceki AP değeri tabloya eklenir.
        row["AP_Delta"] = current["AP"] - prev["AP"]
        # AP farkı hesaplanır.
        row["AP_Gain_%"] = 100 * (current["AP"] - prev["AP"]) / abs(prev["AP"]) if abs(prev["AP"]) > 1e-12 else np.nan
        # AP yüzde değişimi hesaplanır.
        row["Previous_F1"] = prev["F1"]
        # Önceki F1 değeri tabloya eklenir.
        row["F1_Delta"] = current["F1"] - prev["F1"]
        # F1 farkı hesaplanır.
        row["F1_Gain_%"] = 100 * (current["F1"] - prev["F1"]) / abs(prev["F1"]) if abs(prev["F1"]) > 1e-12 else np.nan
        # F1 yüzde değişimi hesaplanır.
        rows.append(row)
        # Karşılaştırma satırı listeye eklenir.
    return pd.DataFrame(rows)
    # Tüm karşılaştırmalar tablo olarak döndürülür.


def make_random_forest():
    """Baseline ve gatekeeper için RandomForest modeli kurar."""
    return RandomForestClassifier(
        n_estimators=300,
        # 300 ağaç, eğitim süresi ve stabilite arasında dengeli bir seçimdir.
        max_features="sqrt",
        # Her bölünmede featureların karekökü kadar aday denenir.
        class_weight="balanced_subsample",
        # Sınıf dengesizliğini her bootstrap örneğinde dengelemeye çalışır.
        random_state=RANDOM_STATE,
        # Tekrarlanabilirlik sağlar.
        n_jobs=-1,
        # Mevcut tüm işlemciler kullanılır.
    )

def resample_indices(y_train, ratio, method):
    """Train set üzerinde over/under sampling için seçilecek indeksleri üretir."""
    if method == "none":
        # Sampling yoksa train set olduğu gibi kullanılır.
        return np.arange(len(y_train))

    rng = np.random.RandomState(RANDOM_STATE)
    # Sampling işleminin tekrarlanabilir olması için sabit random generator oluşturulur.
    pos = np.where(y_train == 1)[0]
    # Pozitif sınıf indexleri alınır.
    neg = np.where(y_train == 0)[0]
    # Negatif sınıf indexleri alınır.
    current_ratio = len(pos) / len(neg)
    # Mevcut pozitif/negatif oranı hesaplanır.

    if method == "oversampling":
        # Oversampling az olan sınıfı çoğaltır.
        if current_ratio < ratio:
            n_pos, n_neg = int(np.ceil(ratio * len(neg))), len(neg)
            # Hedef oran için pozitif sınıf çoğaltılır.
        else:
            n_pos, n_neg = len(pos), int(np.ceil(len(pos) / ratio))
            # Hedef oran için negatif sınıf çoğaltılır.
        selected_pos = rng.choice(pos, n_pos, replace=n_pos > len(pos))
        # Pozitif örnekler gerekirse tekrarlı seçilir.
        selected_neg = rng.choice(neg, n_neg, replace=n_neg > len(neg))
        # Negatif örnekler gerekirse tekrarlı seçilir.

    elif method == "undersampling":
        # Undersampling fazla olan sınıfı azaltır.
        if current_ratio < ratio:
            n_pos = len(pos)
            # Pozitif sınıf korunur.
            n_neg = min(len(neg), max(5, int(np.floor(len(pos) / ratio))))
            # Negatif sınıf hedef orana göre azaltılır.
        else:
            n_pos = min(len(pos), max(5, int(np.floor(ratio * len(neg)))))
            # Pozitif sınıf hedef orana göre azaltılır.
            n_neg = len(neg)
            # Negatif sınıf korunur.
        selected_pos = rng.choice(pos, n_pos, replace=False)
        # Pozitif örnekler tekrarsız seçilir.
        selected_neg = rng.choice(neg, n_neg, replace=False)
        # Negatif örnekler tekrarsız seçilir.

    else:
        raise ValueError("method: none, oversampling veya undersampling olmalı.")

    idx = np.concatenate([selected_pos, selected_neg])
    # Seçilen pozitif ve negatif indexler birleştirilir.
    rng.shuffle(idx)
    # Train sırası karıştırılır.
    return idx
    # Sampling sonrası train indexleri döndürülür.

## 4. Step 5 sonrası karar: yalnızca RandomForest

Step 5 sonucunda iki veri setinde de RandomForest ile devam edilecek.

Bu nedenle bu notebookta:

- Feature selection yapılmaz.
- ExtraTrees veya HistGradientBoosting tekrar denenmez.
- Aynı train/test split korunur.
- Sampling yalnızca train set üzerinde uygulanır.
- Test set doğal dağılımında bırakılır.


In [ ]:
forced_feature_sets = {
    "ERa_BLA_assay": "rdkit",
    "ERa_LUC_VM7_assay": "all_available",
}
# Step 3 sonucuna göre devam edilecek feature setler sabitlenir.

gatekeeper_rows = []
# Her veri seti için sampling öncesi RandomForest gatekeeper sonucu burada tutulur.

for dataset_key in DATASETS:
    # Her veri seti sırayla işlenir.
    df = load_feature_table(dataset_key)
    # Hazır feature CSV dosyası okunur.

    selected_feature_set = forced_feature_sets[dataset_key]
    # Bu veri seti için kullanılacak feature set alınır.
    selected_features = feature_columns(df, selected_feature_set)
    # Seçilen feature setin kolonları alınır.

    X_train, X_test, y_train, y_test, df_train, df_test = split_data(df, selected_features)
    # Aynı random_state ile train/test ayrımı yapılır.

    X_train, X_test = impute_train_test(X_train, X_test)
    # Eksik değerler SimpleImputer ile doldurulur.
    # Imputer median değerleri sadece train setten öğrenir ve sonra test sete uygular.

    model = make_random_forest()
    # Sampling öncesi gatekeeper model RandomForest olarak kurulur.
    model.fit(X_train, y_train)
    # Model doğal train set üzerinde eğitilir.

    y_pred = model.predict(X_test)
    # Doğal test seti için sınıf tahmini alınır.
    y_score = class1_score(model, X_test)
    # ROC/AP için class 1 skoru alınır.

    metrics = metric_dict(y_test, y_pred, y_score)
    # Test performans metrikleri hesaplanır.
    metrics.update({
        "Dataset": dataset_key,
        "Comparison": "Gatekeeper RF",
        "Step": "05_random_forest_gatekeeper",
        "ModelType": "RandomForest",
        "FeatureSet": selected_feature_set,
        "SamplingScenario": "none",
        "SamplingMethod": "none",
        "SamplingRatio": 1.0,
        "FeatureCount": len(selected_features),
        "SelectedFeatures": selected_features,
    })
    # Gatekeeper satırına model, feature ve sampling bilgileri eklenir.

    gatekeeper_rows.append(metrics)
    # Gatekeeper sonucu listeye eklenir.

gatekeeper_df = pd.DataFrame(gatekeeper_rows)
# Gatekeeper sonuçları tabloya çevrilir.

show_table(
    gatekeeper_df[["Dataset", "ModelType", "FeatureSet", "FeatureCount", "ROC", "AP", "F1"]],
    title="Sampling öncesi RandomForest gatekeeper sonuçları"
)
# Sampling öncesi RF performansı ekranda gösterilir.


## 5. RandomForest ile sampling senaryoları

Sampling sadece train set üzerinde uygulanır. Test set doğal dağılımında kalır.

Bu hücrede her veri seti için aynı split kullanılır; sadece train set içindeki sınıf oranı değiştirilir.


In [ ]:
lesson_out = OUTPUT_ROOT / "06_resampling"
# Sampling sonuçları için çıktı klasörü belirlenir.
lesson_out.mkdir(parents=True, exist_ok=True)
# Çıktı klasörü yoksa oluşturulur.

sampling_scenarios = [
    ("none", 1.0, "none"),
    ("balanced_oversampling", 1.0, "oversampling"),
    ("balanced_undersampling", 1.0, "undersampling"),
    ("positive_5x_oversampling", 5.0, "oversampling"),
    ("negative_5x_oversampling", 0.2, "oversampling"),
    ("positive_5x_undersampling", 5.0, "undersampling"),
    ("negative_5x_undersampling", 0.2, "undersampling"),
]
# Denenecek sampling senaryoları tanımlanır.
# Ratio = pozitif/negatif oranıdır.
# ratio=1.0 dengeli, ratio=5.0 pozitif 5 kat fazla, ratio=0.2 negatif 5 kat fazla anlamına gelir.

sampling_rows = []
# Tüm sampling sonuçları burada tutulur.
best_sampling_rows = []
# Her veri seti için en iyi sampling sonucu burada tutulur.
comparison_rows = []
# Gatekeeper vs en iyi sampling karşılaştırma satırları burada tutulur.

for dataset_key in DATASETS:
    # Her veri seti sırayla işlenir.
    df = load_feature_table(dataset_key)
    # Hazır feature CSV dosyası okunur.

    gatekeeper_row = gatekeeper_df[gatekeeper_df["Dataset"] == dataset_key].iloc[0]
    # Bu veri seti için sampling öncesi gatekeeper RF sonucu alınır.
    selected_features = list(gatekeeper_row["SelectedFeatures"])
    # Gatekeeper ile aynı feature listesi kullanılır.

    X_train, X_test, y_train, y_test, df_train, df_test = split_data(df, selected_features)
    # Gatekeeper ile aynı random_state ve aynı split kullanılır.

    X_train, X_test = impute_train_test(X_train, X_test)
    # Eksik değerler SimpleImputer ile doldurulur.
    # Imputer sadece train set üzerinde öğrenilir.

    dataset_rows = []
    # Bu veri setinin sampling sonuçları burada tutulur.

    for scenario_name, ratio, sampling_method in sampling_scenarios:
        # Her sampling senaryosu sırayla denenir.
        sampled_idx = resample_indices(y_train, ratio=ratio, method=sampling_method)
        # Sampling indexleri sadece train set üzerinden üretilir.

        X_train_sampled = X_train[sampled_idx]
        # Sampling uygulanmış train feature matrisi hazırlanır.
        y_train_sampled = y_train[sampled_idx]
        # Sampling uygulanmış train target vektörü hazırlanır.

        sampled_counts = dict(pd.Series(y_train_sampled).value_counts().sort_index())
        # Sampling sonrası train sınıf dağılımı hesaplanır.
        print(f"{dataset_key} | {scenario_name} | sampled train distribution: {sampled_counts}")
        # Sampling sonrası train dağılımı ekrana yazdırılır.

        model = make_random_forest()
        # Bu notebookta yalnızca RandomForest modeli kullanılır.
        model.fit(X_train_sampled, y_train_sampled)
        # Model sampling uygulanmış train set üzerinde eğitilir.

        y_pred = model.predict(X_test)
        # Doğal test seti için sınıf tahmini alınır.
        y_score = class1_score(model, X_test)
        # ROC/AP için class 1 skoru alınır.

        metrics = metric_dict(y_test, y_pred, y_score)
        # Test performans metrikleri hesaplanır.
        metrics.update({
            "Dataset": dataset_key,
            "Step": "06_resampling",
            "ModelType": "RandomForest",
            "FeatureSet": gatekeeper_row["FeatureSet"],
            "SamplingScenario": scenario_name,
            "SamplingMethod": sampling_method,
            "SamplingRatio": ratio,
            "TrainClass0_after_sampling": int(sampled_counts.get(0, 0)),
            "TrainClass1_after_sampling": int(sampled_counts.get(1, 0)),
            "FeatureCount": len(selected_features),
            "SelectedFeatures": selected_features,
        })
        # Sonuç satırına sampling ve train dağılımı bilgileri eklenir.

        dataset_rows.append(metrics)
        # Veri seti sonuç listesine eklenir.
        sampling_rows.append(metrics)
        # Genel sampling sonuç listesine eklenir.

    dataset_df = pd.DataFrame(dataset_rows).sort_values("ROC", ascending=False)
    # Veri seti sampling sonuçları ROC'a göre sıralanır.
    dataset_df["SamplingLabel"] = dataset_df["SamplingScenario"]
    # Grafik için sampling senaryo etiketi oluşturulur.
    dataset_df.to_csv(lesson_out / f"{dataset_key}_rf_sampling_metrics.csv", index=False)
    # Veri seti sampling sonuçları CSV olarak kaydedilir.

    plot_metric_bar(
        dataset_df,
        label_col="SamplingLabel",
        metric="ROC",
        title=f"{dataset_key} — RandomForest sampling ROC karşılaştırması",
        out_file=lesson_out / f"{dataset_key}_rf_sampling_roc.png",
    )
    # Sampling senaryolarının ROC değerleri bar chart olarak çizilir.

    show_table(dataset_df, title=f"{dataset_key} — RandomForest sampling sonuçları")
    # Sampling sonuç tablosu ekranda gösterilir.

    best_sampling = dataset_df.iloc[0].to_dict()
    # ROC-AUC değerine göre en iyi sampling sonucu seçilir.
    best_sampling_rows.append(best_sampling)
    # En iyi sampling sonucu saklanır.

    reference_row = {
        "Dataset": dataset_key,
        "Comparison": "Gatekeeper RF",
        "ModelType": "RandomForest",
        "FeatureSet": gatekeeper_row["FeatureSet"],
        "SamplingScenario": "none",
        "SamplingMethod": "none",
        "SamplingRatio": 1.0,
        "FeatureCount": int(gatekeeper_row["FeatureCount"]),
        "ROC": gatekeeper_row["ROC"],
        "AP": gatekeeper_row["AP"],
        "F1": gatekeeper_row["F1"],
        "ROC_Delta_vs_Gatekeeper": 0.0,
        "ROC_Gain_%": 0.0,
        "AP_Delta_vs_Gatekeeper": 0.0,
        "AP_Gain_%": 0.0,
        "F1_Delta_vs_Gatekeeper": 0.0,
        "F1_Gain_%": 0.0,
    }
    # Karşılaştırma tablosunun ilk satırı sampling öncesi gatekeeper RF sonucudur.

    best_sampling_row = {
        "Dataset": dataset_key,
        "Comparison": "Best RF sampling result",
        "ModelType": "RandomForest",
        "FeatureSet": best_sampling["FeatureSet"],
        "SamplingScenario": best_sampling["SamplingScenario"],
        "SamplingMethod": best_sampling["SamplingMethod"],
        "SamplingRatio": best_sampling["SamplingRatio"],
        "FeatureCount": int(best_sampling["FeatureCount"]),
        "ROC": best_sampling["ROC"],
        "AP": best_sampling["AP"],
        "F1": best_sampling["F1"],
        "ROC_Delta_vs_Gatekeeper": best_sampling["ROC"] - gatekeeper_row["ROC"],
        "ROC_Gain_%": 100 * (best_sampling["ROC"] - gatekeeper_row["ROC"]) / abs(gatekeeper_row["ROC"]) if abs(gatekeeper_row["ROC"]) > 1e-12 else np.nan,
        "AP_Delta_vs_Gatekeeper": best_sampling["AP"] - gatekeeper_row["AP"],
        "AP_Gain_%": 100 * (best_sampling["AP"] - gatekeeper_row["AP"]) / abs(gatekeeper_row["AP"]) if abs(gatekeeper_row["AP"]) > 1e-12 else np.nan,
        "F1_Delta_vs_Gatekeeper": best_sampling["F1"] - gatekeeper_row["F1"],
        "F1_Gain_%": 100 * (best_sampling["F1"] - gatekeeper_row["F1"]) / abs(gatekeeper_row["F1"]) if abs(gatekeeper_row["F1"]) > 1e-12 else np.nan,
    }
    # Karşılaştırma tablosunun ikinci satırı ROC-AUC temelli en iyi RF sampling sonucudur.

    comparison_rows.extend([reference_row, best_sampling_row])
    # İki satırlık karşılaştırma genel listeye eklenir.

sampling_df = pd.DataFrame(sampling_rows)
# Tüm sampling sonuçları tabloya çevrilir.
best_sampling_df = pd.DataFrame(best_sampling_rows)
# En iyi sampling sonuçları tabloya çevrilir.
comparison_df = pd.DataFrame(comparison_rows)
# Gatekeeper ve en iyi sampling karşılaştırmaları tabloya çevrilir.

sampling_df.to_csv(lesson_out / "06_rf_sampling_metrics_all.csv", index=False)
# Tüm RF sampling sonuçları CSV olarak kaydedilir.
best_sampling_df.to_csv(lesson_out / "06_best_rf_sampling_per_dataset.csv", index=False)
# En iyi RF sampling sonuçları CSV olarak kaydedilir.
comparison_df.to_csv(lesson_out / "06_gatekeeper_vs_best_rf_sampling_all.csv", index=False)
# Gatekeeper ve en iyi RF sampling karşılaştırması CSV olarak kaydedilir.

gain_df = add_gain(best_sampling_df, gatekeeper_df, "06_RF_Resampling", "05_RF_Gatekeeper")
# Sampling sonrası en iyi sonuç, sampling öncesi RF gatekeeper ile karşılaştırılır.
gain_df.to_csv(lesson_out / "06_gain_vs_rf_gatekeeper.csv", index=False)
# Kazanç tablosu CSV olarak kaydedilir.

for dataset_key in DATASETS:
    # İki classification veri seti için ayrı gatekeeper vs en iyi sampling tablosu üretilir.
    dataset_comparison_table = comparison_df[comparison_df["Dataset"] == dataset_key].copy()
    # İlgili veri setinin iki satırlık karşılaştırması alınır.
    dataset_comparison_table.to_csv(lesson_out / f"{dataset_key}_gatekeeper_vs_best_rf_sampling.csv", index=False)
    # Veri setine özel karşılaştırma tablosu CSV olarak kaydedilir.

    show_table(
        dataset_comparison_table[
            [
                "Dataset",
                "Comparison",
                "ModelType",
                "FeatureSet",
                "SamplingScenario",
                "SamplingMethod",
                "SamplingRatio",
                "FeatureCount",
                "ROC",
                "AP",
                "F1",
                "ROC_Delta_vs_Gatekeeper",
                "ROC_Gain_%",
                "AP_Delta_vs_Gatekeeper",
                "AP_Gain_%",
                "F1_Delta_vs_Gatekeeper",
                "F1_Gain_%",
            ]
        ],
        title=f"{dataset_key} — Gatekeeper RF vs en iyi RF sampling sonucu"
    )
    # Her veri seti için gatekeeper ve en iyi sampling karşılaştırması ekranda gösterilir.

show_table(
    gain_df[["Dataset", "ModelType", "SamplingScenario", "ROC", "Previous_ROC", "ROC_Delta", "ROC_Gain_%", "AP", "F1"]],
    title="06 — RF gatekeeper sonucuna göre sampling kazancı"
)
# Sampling adımının performans kazancı ekranda gösterilir.
